# Phase 6: Orchestration with LangGraph

So far each agent ran alone. A LangGraph graph connects them: every step reads and writes a shared state, and conditional edges choose the next step.

Flow: triage, routing, web check, duplicate check. Critical reports skip the web check, and a planned outage ends early.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from community_voice.db.repository import reset_database
from community_voice.graph.workflow import build_graph
from community_voice.rag.reports import reset_reports
from community_voice.schemas.report import ReportIn

## 1. The graph

The diagram as text. Paste it into any Mermaid viewer (for example mermaid.live) to see it drawn.

In [ ]:
graph = build_graph(InMemorySaver())
print(graph.get_graph().draw_mermaid())

## 2. Run a report and watch each step

`graph.stream` returns each step's update as soon as it finishes.

Since Phase 7 the graph saves each report and then pauses for a human review, so these runs stop there. Phase 7 shows the review.

In [ ]:
def run(report_id, text):
    inputs = {"report_id": report_id, "report": ReportIn(text=text)}
    config = {"configurable": {"thread_id": report_id}}
    for chunk in graph.stream(inputs, config, stream_mode="updates", version="v2"):
        if chunk["type"] != "updates":
            continue
        for node, update in chunk["data"].items():
            print(f"[{node}]")
            for key, value in update.items():
                print(f"   {key}: {str(value)[:160]}")

In [ ]:
reset_database()
reset_reports()
run(
    "r02",
    "Amanzi ayaphuma emgwaqeni e-Umlazi V Section, "
    "kuvuza ipayipi elikhulu kusukela izolo.",
)

## 3. A critical report

Live wires near a hospital. Look for the web check: it should be skipped.

In [ ]:
run(
    "r05",
    "After last night's storm there are live wires hanging low over the "
    "road near Tembisa Hospital. Children walk under them to get to school.",
)

## 4. A duplicate

The same burst pipe in Umlazi, reported again in different words.

In [ ]:
run(
    "r02b",
    "A big pipe has burst in Umlazi V Section and water has been running "
    "down the street since yesterday.",
)

## What to notice

- One shared state: each step adds its result and later steps read it.
- Different reports take different paths through the same graph.
- The duplicate check compares with earlier reports and asks a model to confirm.
- If the first report hit the planned-outage path, it was not stored, so the duplicate is not found. Try another report.